# Day 3 · 3교시 [실습] ML 파이프라인 — `03_ml_pipeline`

교안 **3.4·3.6·3.7·3.8·3.9** 의 코드가 셀 단위로 그대로 들어 있다.
붙여 넣지 말고 **위에서부터 순서대로 실행**하면 된다.

| 절 | 내용 | 교안 |
|---|---|---|
| 1 | 회귀 성적표 — RMSE·R² | 3.4 |
| 2 | 분류 성적표 — 혼동행렬 | 3.6 |
| 3 | **과적합** — train 점수를 믿지 마라 | 3.7 |
| 4 | **함정** — 정확도가 거짓말할 때 | 3.8 |
| 5 | 스윕 — 어느 깊이가 좋은가 | 3.9 |

> 전부 **오프라인**(sklearn 내장·합성 데이터), **시드 고정**(몇 번 돌려도 같은 숫자).
> 의존성: `pip install scikit-learn numpy`

## 1. 회귀 성적표 — RMSE 와 R² (3.4)

숫자를 맞히는 문제의 성적은 **얼마나 빗나갔나**로 잰다.

In [1]:
import numpy as np
from sklearn.datasets import load_diabetes
from sklearn.linear_model import LinearRegression
from sklearn.metrics import mean_squared_error, r2_score
from sklearn.model_selection import train_test_split

dia = load_diabetes()
X, y = dia.data, dia.target
print("데이터:", X.shape, "| 정답(당뇨 진행도) 범위:", f"{y.min():.0f} ~ {y.max():.0f}")

Xtr, Xte, ytr, yte = train_test_split(X, y, test_size=0.3, random_state=42)
model = LinearRegression().fit(Xtr, ytr)
pred = model.predict(Xte)

rmse = mean_squared_error(yte, pred) ** 0.5
print(f"\nRMSE : {rmse:.1f}   ← 평균 이만큼 빗나간다 (정답과 같은 단위)")
print(f"R2   : {r2_score(yte, pred):.3f}   ← 1.0 이 완벽, 0.0 은 '평균만 찍는 수준'")

baseline = np.full(len(yte), ytr.mean())
print(f"\n비교) 아무것도 안 배우고 '평균'만 답했을 때 RMSE: "
      f"{mean_squared_error(yte, baseline) ** 0.5:.1f}")

데이터: (442, 10) | 정답(당뇨 진행도) 범위: 25 ~ 346

RMSE : 53.1   ← 평균 이만큼 빗나간다 (정답과 같은 단위)
R2   : 0.477   ← 1.0 이 완벽, 0.0 은 '평균만 찍는 수준'

비교) 아무것도 안 배우고 '평균'만 답했을 때 RMSE: 73.7


**읽는 법** — `RMSE 53.1` 은 예측이 평균 53만큼 빗나간다는 뜻이다(정답 범위 25~346).
그런데 아무것도 안 배우고 **평균만 답해도 73.7** 이 나온다. 즉 우리 모델은 *평균 찍기보다
28% 덜 빗나간 정도*다. **`R² 0.477` 이 그 비교를 한 줄로 요약한 숫자**다.
점수는 **baseline 과 비교해야** 의미가 생긴다.

## 2. 분류 성적표 — 혼동행렬 (3.6)

정확도 하나로는 *무엇을 무엇으로 착각했는지* 알 수 없다.

In [2]:
from sklearn.datasets import load_iris
from sklearn.linear_model import LogisticRegression
from sklearn.metrics import accuracy_score, confusion_matrix

iris = load_iris()
Xi, yi = iris.data, iris.target
names = [str(n) for n in iris.target_names]
print("데이터:", Xi.shape, "| 품종:", names)

Xi_tr, Xi_te, yi_tr, yi_te = train_test_split(
    Xi, yi, test_size=0.3, random_state=42, stratify=yi
)
clf = LogisticRegression(max_iter=200).fit(Xi_tr, yi_tr)
ip = clf.predict(Xi_te)
print(f"\n정확도: {accuracy_score(yi_te, ip):.3f}")

cm = confusion_matrix(yi_te, ip)
w = max(len(n) for n in names)
print("\n혼동행렬 (행=실제, 열=예측):")
print(" " * (w + 2) + "  ".join(f"{n:>{w}}" for n in names))
for i, row in enumerate(cm):
    print(f"{names[i]:>{w}}  " + "  ".join(f"{v:>{w}}" for v in row))

데이터: (150, 4) | 품종: ['setosa', 'versicolor', 'virginica']

정확도: 0.933

혼동행렬 (행=실제, 열=예측):
                setosa  versicolor   virginica
    setosa          15           0           0
versicolor           0          14           1
 virginica           0           2          13


**읽는 법** — 대각선이 정답, 나머지가 오답이다.
`setosa` 는 15개 전부 맞혔고, 틀린 3건은 **전부 `versicolor ↔ virginica` 사이**에서 났다.
우연이 아니라 **그 두 품종이 실제로 비슷하다**는 뜻이다 — 성능을 올리려면 저 둘을
가르는 특성을 찾아야 한다. 정확도 `0.933` 만 봤다면 몰랐을 정보다.

## 3. 과적합 — train 점수가 높은 게 좋은 모델이 아니다 (3.7)

실무 데이터의 라벨에는 **오류가 섞여 있다**(잘못 분류된 불량품, 오진, 오타).
붓꽃 라벨 일부를 일부러 어지럽혀 놓고, 복잡한 모델부터 단순한 모델까지 돌려 본다.

In [3]:
from sklearn.tree import DecisionTreeClassifier

rng = np.random.default_rng(0)
y_dirty = yi.copy()
flip = rng.choice(150, size=25, replace=False)   # 150개 중 25개를 골라
y_dirty[flip] = rng.integers(0, 3, size=25)      # 라벨을 무작위로 바꿔 버린다
print(f"라벨 오염: 150개 중 {(y_dirty != yi).sum()}개가 실제로 바뀜")

Xd_tr, Xd_te, yd_tr, yd_te = train_test_split(
    Xi, y_dirty, test_size=0.3, random_state=42, stratify=yi
)

print(f"\n{'max_depth':>10} | {'train':>7} | {'test':>7} | 격차")
print("-" * 42)
for d in [None, 5, 3, 2]:                        # None = 깊이 제한 없음(가장 복잡)
    t = DecisionTreeClassifier(max_depth=d, random_state=0).fit(Xd_tr, yd_tr)
    a = accuracy_score(yd_tr, t.predict(Xd_tr))
    b = accuracy_score(yd_te, t.predict(Xd_te))
    print(f"{str(d):>10} | {a:>7.3f} | {b:>7.3f} | {a - b:+.3f}")

라벨 오염: 150개 중 18개가 실제로 바뀜

 max_depth |   train |    test | 격차
------------------------------------------
      None |   1.000 |   0.689 | +0.311
         5 |   0.943 |   0.711 | +0.232
         3 |   0.886 |   0.800 | +0.086
         2 |   0.876 |   0.800 | +0.076


**`train 1.000` 짜리가 가장 나쁜 모델이었다.**

| | train | test | |
|---|---|---|---|
| `None` | **1.000** | **0.689** | 오염된 라벨까지 통째로 **외웠다** |
| `3` | 0.886 | **0.800** | train 은 낮은데 **test 는 더 높다** |

깊이 제한이 없으면 트리는 잘못 붙은 라벨까지 하나하나 외운다 — 실력이 아니라 **암기**이고,
처음 보는 데이터 앞에서 무너진다. 이게 **과적합**이며, **train 점수가 아니라 train-test
격차**를 봐야 하는 이유다. 데이터를 train/test 로 나누지 않았다면 `1.000` 을 보고
"완벽한 모델!"이라고 발표했을 것이다.

## 4. 🔥 함정 — 정확도가 거짓말할 때 (3.8)

찾아야 할 대상이 **아주 드문** 문제(불량품·희귀병·이상거래)에서는 정확도가 거짓말을 한다.
아무것도 학습하지 않고 **"전부 정상"이라고만 찍는 모델**의 성적표를 보자.

In [4]:
from sklearn.dummy import DummyClassifier
from sklearn.metrics import f1_score, recall_score

rng2 = np.random.default_rng(7)
Xb = rng2.normal(size=(1000, 5))
yb = (rng2.random(1000) < 0.02).astype(int)      # 양성(불량)이 2% 안팎인 데이터

Xb_tr, Xb_te, yb_tr, yb_te = train_test_split(Xb, yb, test_size=0.3, random_state=0)

dummy = DummyClassifier(strategy="most_frequent").fit(Xb_tr, yb_tr)  # 무조건 다수 클래스
dp = dummy.predict(Xb_te)

print(f"양성 비율: {yb.mean():.1%}  (불량품·질병·이상거래 같은 희귀 사건)")
print("\n'전부 정상'이라고만 찍는 모델의 성적표:")
print(f"  accuracy = {accuracy_score(yb_te, dp):.3f}   ← 훌륭해 보인다")
print(f"  recall   = {recall_score(yb_te, dp, zero_division=0):.3f}   ← 찾아야 할 걸 하나도 못 잡았다")
print(f"  f1       = {f1_score(yb_te, dp, zero_division=0):.3f}")

양성 비율: 2.6%  (불량품·질병·이상거래 같은 희귀 사건)

'전부 정상'이라고만 찍는 모델의 성적표:
  accuracy = 0.983   ← 훌륭해 보인다
  recall   = 0.000   ← 찾아야 할 걸 하나도 못 잡았다
  f1       = 0.000


**`정확도 98.3%`** — 학습을 한 글자도 안 한 모델이다. 양성이 2.6%뿐이니
전부 "정상"이라고만 찍어도 97% 이상은 자동으로 맞는다.
그런데 **`recall 0.000`** — **정작 잡아야 할 것을 하나도 못 잡았다.**

| 상황 | 봐야 할 지표 |
|---|---|
| 놓치면 큰일 (암 진단, 불량 검출) | **recall** |
| 잘못 잡으면 곤란 (스팸 분류) | **precision** |
| 둘 다 중요 | **f1** |

> ⚠️ **에이전트에게 지표를 지정해 주지 않으면 대개 accuracy 를 보고한다.**
> *"정확도 98% 나왔습니다"* 를 그대로 믿으면 안 되는 이유다.

## 5. 스윕 — 어느 깊이가 좋은가 (3.9)

3의 오염 데이터를 그대로 쓴다. **가운데가 가장 높은 산 모양**이 나온다.

In [5]:
print(f"{'max_depth':>10} | {'test acc':>8}")
print("-" * 23)
for d in [1, 2, 3, 5, None]:
    t = DecisionTreeClassifier(max_depth=d, random_state=0).fit(Xd_tr, yd_tr)
    print(f"{str(d):>10} | {accuracy_score(yd_te, t.predict(Xd_te)):>8.3f}")

 max_depth | test acc
-----------------------
         1 |    0.556
         2 |    0.800
         3 |    0.800
         5 |    0.711
      None |    0.689


- `1`: **너무 단순해서** 못 배웠다 → **과소적합**
- `2~3`: 딱 좋다
- `5~None`: **너무 복잡해서** 외워 버렸다 → **과적합**

**모델은 크다고 좋은 게 아니라 "알맞아야" 좋다.** 그리고 꼭대기가 어디인지는
**돌려 봐야만 안다** — 그래서 반복이 필요하고, 그 반복이 에이전트에게 맡길 일이다.

> ⚠️ test 가 45개뿐이라 `0.800` 과 `0.778` 의 차이는 **1건**이다. 그 차이로 우열을 논하면 안 된다.

---

## 정리

1. **baseline 과 비교하라** — `R² 0.477` = "평균 찍기보다 조금 나은 정도"
2. **혼동행렬을 보라** — 정확도 뒤에 *"versicolor와 virginica를 헷갈린다"* 가 숨어 있었다
3. **train 점수를 믿지 마라** — `train 1.000` 짜리가 가장 나빴다
4. **지표를 문제에 맞게 골라라** — `accuracy 0.983 / recall 0.000`

> **AI 에게 일을 시키는 능력과, 그 결과를 판단하는 능력은 다른 능력이다.**